In [2]:
import re
import json
from pathlib import Path

INPUT_TXT = Path("../data/processed/turkce.txt")
OUTPUT_CLEAN_TXT = Path("../data/processed/turkce_clean.txt")
OUTPUT_JSON = Path("../data/processed/turkce_questions.json")

with open(INPUT_TXT, "r", encoding="utf-8") as f:
    raw_text = f.read()

def basic_clean(t: str) -> str:
    t = t.replace("\r", "")
    t = t.replace("\xa0", " ")

    # Eğer txt üretirken sayfa ayraçları bastıysan (===== PAGE X =====) onları kaldır
    t = re.sub(r"(?mi)^\s*={3,}\s*PAGE\s*\d+\s*={3,}\s*$", "", t)

    # Header/footer gürültüleri
    t = re.sub(r"(?mi)^\s*DİĞER SAYFAYA GEÇİNİZ\.?\s*$", "", t)
    t = re.sub(r"(?mi)^\s*GENEL YETENEK TESTİ\s*$", "", t)
    t = re.sub(r"(?mi)^\s*GENEL YETENEK\s*$", "", t)
    t = re.sub(r"(?mi)^\s*TEMEL SORU KİTAPÇIĞI\s*$", "", t)
    t = re.sub(r"(?mi)^\s*SORU KİTAPÇIĞI\s*$", "", t)

    # Tek başına sayfa numarası gibi duran satırlar
    t = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", t)

    # Çoklu boş satır
    t = re.sub(r"\n{3,}", "\n\n", t)

    return t.strip()

def normalize_questions_and_options(t: str) -> str:
    """
    Parse kolay olsun diye:
    - Soru numaralarını satır başına iter: \n12. ...
    - Şıkları satır başına iter: \nA) ...
    - A. / A- -> A) formatına yaklaştırır
    """
    # soru numarası satır içine kaydıysa satır başına çek
    t = re.sub(r"(?<!\n)(\d{1,2})\.\s+", r"\n\1. ", t)

    # Şıkları satır başına çek
    t = re.sub(r"(?<!\n)\s([A-E])\)", r"\n\1) ", t)
    t = re.sub(r"(?<!\n)\s([A-E])\.", r"\n\1) ", t)   # A. -> A)
    t = re.sub(r"(?<!\n)\s([A-E])\-", r"\n\1) ", t)   # A- -> A)

    # "A )" gibi boşluklu şık işaretlerini düzelt
    t = re.sub(r"(?m)^\s*([A-E])\s*\)\s*", r"\1) ", t)

    # "12 ." gibi bozuk soru numarası satırlarını düzelt
    t = re.sub(r"(?m)^\s*(\d{1,2})\s*\.\s*", r"\1. ", t)

    # Fazla boşlukları azalt
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)

    return t.strip()

YEAR_PATTERNS = [
    r"KPSS[^\n]{0,60}/\s*(20\d{2})",
    r"(20\d{2})\s*-\s*KPSS",
    r"(20\d{2})-KPSS",
    r"ÖN\s*LİSANS\s*/\s*(20\d{2})",
    r"ÖNLİSANS\s*/\s*(20\d{2})",
]
YEAR_REGEX = re.compile("|".join(f"(?:{p})" for p in YEAR_PATTERNS), flags=re.IGNORECASE)

def mark_year_headers(t: str) -> str:
    """
    Yıl geçen yerleri işaretleyelim ki bölmek kolay olsun.
    """
    def repl(m):
        # eşleşen ilk dolu grubu bul
        year = None
        for g in m.groups():
            if g and re.fullmatch(r"20\d{2}", g):
                year = g
                break
        return f"\n\n### YEAR: {year} ###\n\n" + m.group(0)

    # yıl tespiti için: metinde “KPSS” geçen yıl cümlelerini işaretler
    t2 = YEAR_REGEX.sub(repl, t)
    t2 = re.sub(r"\n{3,}", "\n\n", t2)
    return t2.strip()

text = basic_clean(raw_text)
text = normalize_questions_and_options(text)
text = mark_year_headers(text)

# temizlenmiş txt kaydet
OUTPUT_CLEAN_TXT.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_CLEAN_TXT, "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Temiz TXT:", OUTPUT_CLEAN_TXT)

QSTART = re.compile(r"(?m)^\s*(\d{1,2})\.\s+")       # satır başı: 1. 2. ...
OPTSTART = re.compile(r"(?m)^\s*([A-E])\)\s+")       # satır başı: A) B) ...

def split_by_year_marked(t: str):
    """
    ### YEAR: 2016 ### işaretlerine göre böl.
    Eğer yıl yoksa tek parça döner.
    """
    parts = re.split(r"(?m)^\s*###\s*YEAR:\s*(20\d{2})\s*###\s*$", t)
    # parts: [prefix, year, body, year, body, ...]
    if len(parts) == 1:
        return [(None, t)]

    out = []
    prefix = parts[0].strip()
    if prefix:
        out.append((None, prefix))

    for i in range(1, len(parts), 2):
        year = int(parts[i])
        body = parts[i+1].strip()
        out.append((year, body))
    return out

def split_questions(section_text: str):
    starts = list(QSTART.finditer(section_text))
    if not starts:
        return []
    blocks = []
    for i, m in enumerate(starts):
        qno = int(m.group(1))
        start = m.start()
        end = starts[i+1].start() if i+1 < len(starts) else len(section_text)
        blk = section_text[start:end].strip()
        if len(blk) > 30:
            blocks.append((qno, blk))
    return blocks

def parse_question_block(block: str):
    # "12. " kısmını at
    block = re.sub(r"(?m)^\s*\d{1,2}\.\s+", "", block).strip()

    # cevap varsa yakala (çoğu zaman yok)
    m_ans = re.search(r"(?i)Cevap:\s*([A-E])", block)
    answer = m_ans.group(1).upper() if m_ans else None

    opts = list(OPTSTART.finditer(block))
    if not opts:
        qtext = re.sub(r"\s+", " ", block).strip()
        return qtext, {}, answer

    qtext = block[:opts[0].start()].strip()
    qtext = re.sub(r"\s+", " ", qtext).strip()

    options = {}
    for i, m in enumerate(opts):
        key = m.group(1).upper()
        start = m.end()
        end = opts[i+1].start() if i+1 < len(opts) else len(block)
        val = block[start:end].strip()
        val = re.sub(r"\s+", " ", val).strip()
        options[key] = val

    return qtext, options, answer

records = []
sections = split_by_year_marked(text)

for year, sec in sections:
    qblocks = split_questions(sec)
    for qno, blk in qblocks:
        soru, secenekler, cevap = parse_question_block(blk)

        rec = {
            "id": f"{year}_{qno}" if year else f"unknown_{qno}",
            "yil": year,
            "ders": "Türkçe",
            "soru_no": qno,
            "soru": soru,
            "secenekler": secenekler,
            "cevap": cevap
        }
        records.append(rec)

# JSON kaydet
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"✅ {len(records)} soru işlendi ve kaydedildi: {OUTPUT_JSON}")

# hızlı kontrol
records[:2]


✅ Temiz TXT: ..\data\processed\turkce_clean.txt
✅ 257 soru işlendi ve kaydedildi: ..\data\processed\turkce_questions.json


[{'id': 'unknown_1',
  'yil': None,
  'ders': 'Türkçe',
  'soru_no': 1,
  'soru': 'Aşağıdaki cümlelerin hangisinde, ayraç içindeki sözün anlamını içeren bir söz yoktur?',
  'secenekler': {'A': 'Valizini alelacele toplayıp yola çıktı. (çabucak)',
   'B': 'Kaleden bakınca, göz alabildiğine uzanan yemyeşil ova görünüyordu. (sınırsızcasına)',
   'C': 'Sevmenin verdiği özlemle dolup taşmalısın. (gereğinden çok)',
   'D': 'Acılar da sevinçler gibidir, olgunlaştırır insanı. (sakınmak)',
   'E': 'Eğlenceyi de tadında bırakmak gerekir. (aşırılığa kaçmamak)'},
  'cevap': None},
 {'id': 'unknown_2',
  'yil': None,
  'ders': 'Türkçe',
  'soru_no': 2,
  'soru': 'Büyük şair, iyi roman, kötü öykü gibi ayrımlar yap- mak, hiçbir sanat anlayışına sığmaz. Gerçek eleştiri anlayışında bu en iyi, bu en kötü, diye çürük-sağlam ayırırcasına bir eleştiri yoktur. Yapılması gereken, yaratıcılığını, gücünü, özgünlüğünü değerlendirerek herkesi kendi yerine oturtmaktır. Bu parçada geçen “herkesi kendi yerine oturtm